# Daphnet FoG Detection — Improved Pipeline
## Part 3: Fusion Techniques & Results

Compares 3 fusion techniques:
1. Feature-level fusion (all sensors concatenated)
2. Decision-level stacking (per-sensor base + LR meta-learner)
3. Weighted soft voting (optimised weights)

In [ ]:
def get_sensor_features(X: pd.DataFrame, sensor: str) -> pd.DataFrame:
    """Get feature columns belonging to a specific sensor."""
    # Channels: ankle=0,1,2  thigh=3,4,5  trunk=6,7,8  mag_ankle=9  mag_thigh=10  mag_trunk=11
    sensor_map = {"ankle": [0, 1, 2, 9], "thigh": [3, 4, 5, 10], "trunk": [6, 7, 8, 11]}
    ch_ids = sensor_map[sensor]
    cols = [c for c in X.columns if any(f"ch{ch}_" in c for ch in ch_ids)]
    return X[cols] if cols else X

In [ ]:
def run_fusion_evaluation(features: Dict):
    """Run 3 fusion techniques evaluation. No test-set leakage."""
    from sklearn.model_selection import train_test_split
    subjects = sorted(features.keys())

    fusion_results = {"feature_level": [], "stacking": [], "weighted_voting": []}

    for test_sid in tqdm(subjects, desc="Fusion LOSO"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)
        has_both = len(np.unique(y_test)) == 2

        # ── Fusion 1: Feature-Level ──
        X_tr_p, X_te_p, _, _ = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        X_tr_sm, y_tr_sm = X_tr_p, y_train
        if HAS_SMOTE and np.sum(y_train == 1) >= 6:
            try:
                k_n = min(5, np.sum(y_train == 1) - 1)
                sm = SMOTE(random_state=SEED, k_neighbors=k_n)
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_p, y_train)
            except Exception:
                pass

        clf_fl = RandomForestClassifier(n_estimators=300, max_depth=20, class_weight="balanced",
                                         random_state=SEED, n_jobs=-1)
        clf_fl.fit(X_tr_sm, y_tr_sm)
        y_prob_fl = clf_fl.predict_proba(X_te_p)[:, 1]

        # Threshold on validation split (not test)
        if len(y_train) > 20 and len(np.unique(y_train)) > 1:
            _, X_val_fl, _, y_val_fl = train_test_split(X_tr_p, y_train, test_size=0.2,
                                                         stratify=y_train, random_state=SEED)
            y_val_prob_fl = clf_fl.predict_proba(X_val_fl)[:, 1]
            thr_fl = youden_threshold(y_val_fl, y_val_prob_fl)
        else:
            thr_fl = 0.5

        y_pred_fl = (y_prob_fl >= thr_fl).astype(int)
        m_fl = compute_metrics(y_test, y_pred_fl, y_prob_fl)
        m_fl["subject"] = test_sid
        m_fl["has_both_classes"] = has_both
        fusion_results["feature_level"].append(m_fl)

        # ── Fusion 2 & 3: Per-sensor models with OOF predictions ──
        sensor_oof = {}     # OOF predictions for stacking (no leakage)
        sensor_test = {}    # Test predictions

        for sensor_name in ["ankle", "thigh", "trunk"]:
            X_tr_s = get_sensor_features(X_train, sensor_name)
            X_te_s = get_sensor_features(X_test, sensor_name)

            if X_tr_s.shape[1] == 0:
                continue

            X_tr_s = X_tr_s.replace([np.inf, -np.inf], np.nan)
            X_te_s = X_te_s.replace([np.inf, -np.inf], np.nan)

            vt = VarianceThreshold(threshold=0.0)
            try:
                X_tr_s_vt = pd.DataFrame(vt.fit_transform(X_tr_s))
                X_te_s_vt = pd.DataFrame(vt.transform(X_te_s))
            except Exception:
                continue
            if X_tr_s_vt.shape[1] == 0:
                continue

            imp = KNNImputer(n_neighbors=5)
            X_tr_s_i = imp.fit_transform(X_tr_s_vt)
            X_te_s_i = imp.transform(X_te_s_vt)

            sc = RobustScaler()
            X_tr_s_sc = sc.fit_transform(X_tr_s_i)
            X_te_s_sc = sc.transform(X_te_s_i)

            k_s = min(20, X_tr_s_sc.shape[1])
            sel = SelectKBest(mutual_info_classif, k=k_s)
            X_tr_s_sel = sel.fit_transform(X_tr_s_sc, y_train)
            X_te_s_sel = sel.transform(X_te_s_sc)

            test_probs, oof_probs = build_base_model(X_tr_s_sel, y_train, X_te_s_sel, seed=SEED)
            sensor_test[sensor_name] = test_probs
            sensor_oof[sensor_name] = oof_probs

        if len(sensor_test) < 2:
            continue

        sensor_names = list(sensor_test.keys())
        P_test = np.column_stack([sensor_test[s] for s in sensor_names])
        P_oof = np.column_stack([sensor_oof[s] for s in sensor_names])

        # ── Fusion 2: Stacking with OOF predictions (no leakage) ──
        meta_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
        meta_clf.fit(P_oof, y_train)  # trained on OOF, not in-sample
        y_prob_stack = meta_clf.predict_proba(P_test)[:, 1]

        # Threshold on OOF validation
        y_oof_stack = meta_clf.predict_proba(P_oof)[:, 1]
        thr_st = youden_threshold(y_train, y_oof_stack)
        y_pred_stack = (y_prob_stack >= thr_st).astype(int)
        m_st = compute_metrics(y_test, y_pred_stack, y_prob_stack)
        m_st["subject"] = test_sid
        m_st["has_both_classes"] = has_both
        fusion_results["stacking"].append(m_st)

        # ── Fusion 3: Weighted Voting — weights optimized on OOF (not test) ──
        def neg_f1_weighted(w):
            w = np.abs(w)
            w = w / (w.sum() + 1e-12)
            fused = P_oof @ w
            preds = (fused >= 0.5).astype(int)
            return -f1_score(y_train, preds, zero_division=0)

        w0 = np.ones(len(sensor_names)) / len(sensor_names)
        try:
            res = minimize(neg_f1_weighted, w0, method="Nelder-Mead",
                          options={"maxiter": 200})
            w_opt = np.abs(res.x)
            w_opt = w_opt / (w_opt.sum() + 1e-12)
        except Exception:
            w_opt = w0

        y_prob_wv = P_test @ w_opt
        # Threshold on OOF validation
        y_oof_wv = P_oof @ w_opt
        thr_wv = youden_threshold(y_train, y_oof_wv)
        y_pred_wv = (y_prob_wv >= thr_wv).astype(int)
        m_wv = compute_metrics(y_test, y_pred_wv, y_prob_wv)
        m_wv["subject"] = test_sid
        m_wv["has_both_classes"] = has_both
        m_wv["weights"] = {s: round(w, 3) for s, w in zip(sensor_names, w_opt)}
        fusion_results["weighted_voting"].append(m_wv)

    return fusion_results

In [ ]:
# Step 5: Fusion evaluation
log.info("Running fusion evaluation...")
fusion_results = run_fusion_evaluation(features)

## Fusion Technique Results

In [ ]:
print_fusion_results(fusion_results)

## Advanced Stacking Configurations

6 stacking configurations using different meta-classifiers and weak learners.
The meta-dataset for the meta-classifier includes, **per sensor and per weak learner**:
- Predicted probability (continuous)
- Predicted label (binary)

The target is the correct label.

| Config | Meta-Classifier | Weak Learners |
|--------|----------------|---------------|
| 1 | RandomForest | LightGBM + XGBoost |
| 2 | XGBoost | LightGBM + RandomForest |
| 3 | LightGBM | RandomForest + XGBoost |

In [ ]:
from lightgbm import LGBMClassifier

# ── Weak-learner factories ──────────────────────────────────────────────────
def make_rf(seed=SEED):
    return RandomForestClassifier(
        n_estimators=200, max_depth=20,
        class_weight="balanced", random_state=seed, n_jobs=-1)

def make_xgb(seed=SEED):
    return XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=5, eval_metric="logloss",
        random_state=seed, n_jobs=-1, verbosity=0)

def make_lgbm(seed=SEED):
    return LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        class_weight="balanced", random_state=seed,
        n_jobs=-1, verbose=-1)


def build_weak_model_with_preds(X_tr, y_tr, X_te, clf_factory, seed=SEED):
    """Train a weak classifier with OOF.
    Returns (test_probs, test_preds, oof_probs, oof_preds)."""
    from sklearn.model_selection import StratifiedKFold as SKF

    X_tr = np.asarray(X_tr, dtype=np.float64)
    X_te = np.asarray(X_te, dtype=np.float64)

    oof_probs = np.zeros(len(y_tr))
    oof_preds = np.zeros(len(y_tr))
    inner_cv = SKF(n_splits=3, shuffle=True, random_state=seed)

    for tr_idx, val_idx in inner_cv.split(X_tr, y_tr):
        X_f, y_f = X_tr[tr_idx], y_tr[tr_idx]
        X_v = X_tr[val_idx]

        if HAS_SMOTE and np.sum(y_f == 1) >= 6:
            try:
                k_n = min(5, int(np.sum(y_f == 1)) - 1)
                sm = SMOTE(random_state=seed, k_neighbors=k_n)
                X_f, y_f = sm.fit_resample(X_f, y_f)
            except Exception:
                pass

        clf = clf_factory(seed)
        clf.fit(X_f, y_f)
        oof_probs[val_idx] = clf.predict_proba(X_v)[:, 1]
        oof_preds[val_idx] = clf.predict(X_v)

    # Full model for test predictions
    X_tr_sm, y_tr_sm = X_tr, y_tr
    if HAS_SMOTE and np.sum(y_tr == 1) >= 6:
        try:
            k_n = min(5, int(np.sum(y_tr == 1)) - 1)
            sm = SMOTE(random_state=seed, k_neighbors=k_n)
            X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
        except Exception:
            pass

    clf_full = clf_factory(seed)
    clf_full.fit(X_tr_sm, y_tr_sm)
    test_probs = clf_full.predict_proba(X_te)[:, 1]
    test_preds = clf_full.predict(X_te)

    return test_probs, test_preds, oof_probs, oof_preds


# ── Stacking configurations ─────────────────────────────────────────────────
STACKING_CONFIGS = [
    {"name": "Stack_RF_meta(LGBM+XGB)",  "meta": make_rf,   "weak": [("LGBM", make_lgbm), ("XGB", make_xgb)]},
    {"name": "Stack_XGB_meta(LGBM+RF)",  "meta": make_xgb,  "weak": [("LGBM", make_lgbm), ("RF",  make_rf)]},
    {"name": "Stack_LGBM_meta(RF+XGB)",  "meta": make_lgbm, "weak": [("RF",   make_rf),   ("XGB", make_xgb)]},
]

print(f"Defined {len(STACKING_CONFIGS)} stacking configurations")

In [ ]:
def run_advanced_stacking(features: Dict):
    """Run all stacking configurations with LOSO. Meta-dataset = probs + preds per sensor per weak learner."""
    subjects = sorted(features.keys())
    config_results = {cfg["name"]: [] for cfg in STACKING_CONFIGS}

    for test_sid in tqdm(subjects, desc="Adv. Stacking LOSO"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)
        has_both = len(np.unique(y_test)) == 2

        # ── Pre-process per-sensor features once ──
        sensor_data = {}  # sensor_name -> (X_tr_sel, X_te_sel)
        for sensor_name in ["ankle", "thigh", "trunk"]:
            X_tr_s = get_sensor_features(X_train, sensor_name)
            X_te_s = get_sensor_features(X_test, sensor_name)

            if X_tr_s.shape[1] == 0:
                continue

            X_tr_s = X_tr_s.replace([np.inf, -np.inf], np.nan)
            X_te_s = X_te_s.replace([np.inf, -np.inf], np.nan)

            vt = VarianceThreshold(threshold=0.0)
            try:
                X_tr_s_vt = pd.DataFrame(vt.fit_transform(X_tr_s))
                X_te_s_vt = pd.DataFrame(vt.transform(X_te_s))
            except Exception:
                continue
            if X_tr_s_vt.shape[1] == 0:
                continue

            imp = KNNImputer(n_neighbors=5)
            X_tr_s_i = imp.fit_transform(X_tr_s_vt)
            X_te_s_i = imp.transform(X_te_s_vt)

            sc = RobustScaler()
            X_tr_s_sc = sc.fit_transform(X_tr_s_i)
            X_te_s_sc = sc.transform(X_te_s_i)

            k_s = min(20, X_tr_s_sc.shape[1])
            sel = SelectKBest(mutual_info_classif, k=k_s)
            X_tr_s_sel = sel.fit_transform(X_tr_s_sc, y_train)
            X_te_s_sel = sel.transform(X_te_s_sc)

            sensor_data[sensor_name] = (X_tr_s_sel, X_te_s_sel)

        if len(sensor_data) < 2:
            continue

        # ── Evaluate each stacking configuration ──
        for cfg in STACKING_CONFIGS:
            oof_meta_parts = []
            test_meta_parts = []

            for sensor_name, (X_tr_s, X_te_s) in sensor_data.items():
                for weak_name, weak_factory in cfg["weak"]:
                    te_prob, te_pred, oof_prob, oof_pred = build_weak_model_with_preds(
                        X_tr_s, y_train, X_te_s, weak_factory, seed=SEED)
                    # Each weak learner contributes prob + pred per sensor
                    oof_meta_parts.append(oof_prob)
                    oof_meta_parts.append(oof_pred)
                    test_meta_parts.append(te_prob)
                    test_meta_parts.append(te_pred)

            # Build meta-dataset: columns = [prob_w1_s1, pred_w1_s1, prob_w2_s1, ..., for all sensors]
            P_oof = np.column_stack(oof_meta_parts)
            P_test = np.column_stack(test_meta_parts)

            # Train meta-classifier on OOF meta-dataset (no leakage)
            meta_clf = cfg["meta"](SEED)
            meta_clf.fit(P_oof, y_train)

            y_prob_meta = meta_clf.predict_proba(P_test)[:, 1]

            # Threshold optimisation on OOF
            y_oof_meta = meta_clf.predict_proba(P_oof)[:, 1]
            thr = youden_threshold(y_train, y_oof_meta)
            y_pred_meta = (y_prob_meta >= thr).astype(int)

            m = compute_metrics(y_test, y_pred_meta, y_prob_meta)
            m["subject"] = test_sid
            m["has_both_classes"] = has_both
            m["threshold"] = thr
            config_results[cfg["name"]].append(m)

    return config_results

In [ ]:
log.info("Running advanced stacking configurations...")
stacking_config_results = run_advanced_stacking(features)

### Advanced Stacking Results Comparison

In [ ]:
# ── Print advanced stacking comparison table ──
print(f"\n{'='*100}")
print(f"  ADVANCED STACKING CONFIGURATIONS — DAPHNET")
print(f"{'='*100}")
print(f"  Meta-dataset per fold: probs + predicted labels from each weak learner × each sensor")
print(f"  Sensors: ankle, thigh, trunk | Weak learners: 2 per config")
print(f"  Meta-features: 3 sensors × 2 weak × 2 (prob+pred) = 12 columns")
print(f"{'='*100}")
print(f"{'Configuration':<30} {'F1(agg)':>8} {'F1(mean)':>9} {'Recall':>8} {'Prec':>8} "
      f"{'Spec':>8} {'BalAcc':>8} {'AUC':>8} {'Folds':>6}")
print("-" * 100)

stacking_rows = []
for cfg_name, results in stacking_config_results.items():
    if results:
        agg = aggregate_results(results, cfg_name)
        stacking_rows.append(agg)

stacking_rows.sort(key=lambda x: x.get("f1_agg", 0), reverse=True)

for agg in stacking_rows:
    print(f"{agg['name']:<30} {agg['f1_agg']:>8.4f} {agg.get('f1_mean', 0):>9.4f} "
          f"{agg.get('recall_mean', 0):>8.4f} {agg.get('precision_mean', 0):>8.4f} "
          f"{agg.get('specificity_mean', 0):>8.4f} {agg.get('balanced_acc_mean', 0):>8.4f} "
          f"{agg.get('auc_mean', 0):>8.4f} {agg['n_folds_evaluable']:>3}/{agg['n_folds_total']}")

print("-" * 100)

# ── Compare ALL fusion methods (original + advanced) ──
print(f"\n{'='*100}")
print(f"  COMPLETE FUSION COMPARISON (Original + Advanced Stacking)")
print(f"{'='*100}")

all_fusion = {}
fusion_names_map = {"feature_level": "Feature-Level Fusion",
                    "stacking": "Stacking (LR meta, RF weak)",
                    "weighted_voting": "Weighted Soft Voting"}
for key, results in fusion_results.items():
    if results:
        all_fusion[fusion_names_map.get(key, key)] = results
for cfg_name, results in stacking_config_results.items():
    if results:
        all_fusion[cfg_name] = results

print(f"{'Method':<35} {'F1(agg)':>8} {'F1(mean)':>9} {'Recall':>8} {'Prec':>8} "
      f"{'Spec':>8} {'AUC':>8} {'Folds':>6}")
print("-" * 100)

all_rows = []
for name, results in all_fusion.items():
    agg = aggregate_results(results, name)
    all_rows.append(agg)
all_rows.sort(key=lambda x: x.get("f1_agg", 0), reverse=True)

for agg in all_rows:
    print(f"{agg['name']:<35} {agg['f1_agg']:>8.4f} {agg.get('f1_mean', 0):>9.4f} "
          f"{agg.get('recall_mean', 0):>8.4f} {agg.get('precision_mean', 0):>8.4f} "
          f"{agg.get('specificity_mean', 0):>8.4f} {agg.get('auc_mean', 0):>8.4f} "
          f"{agg['n_folds_evaluable']:>3}/{agg['n_folds_total']}")

print("-" * 100)
print(f"\nBEST OVERALL FUSION METHOD:")
best = all_rows[0]
print(f"  {best['name']} — F1(agg)={best['f1_agg']:.4f}, AUC={best.get('auc_mean', 0):.4f}")

## Save Results

In [ ]:
# Step 6: Save results
save_data = {}
for name, results in all_results.items():
    save_data[name] = aggregate_results(results, name)
save_data["fusion"] = {}
for key, results in fusion_results.items():
    if results:
        save_data["fusion"][key] = aggregate_results(results, key)

# Include advanced stacking configs
save_data["advanced_stacking"] = {}
for cfg_name, results in stacking_config_results.items():
    if results:
        save_data["advanced_stacking"][cfg_name] = aggregate_results(results, cfg_name)

results_file = OUTPUT_DIR / "results_summary.json"

# Convert numpy types for JSON serialization
def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(results_file, "w") as f:
    json.dump(save_data, f, indent=2, default=_convert)
log.info("Results saved to %s", results_file)

elapsed = time.time() - t0
print(f"\nTotal execution time: {elapsed/60:.1f} minutes")